# Modul 03: Datenstrukturen, Quellen und Qualität | Lösungen

## Überblick

Sie strukturieren Beobachtungen, Merkmale und Zielwerte, laden kleine Daten aus mehreren Formaten und erstellen einen nachvollziehbaren Validierungsbericht. Der Schwerpunkt liegt auf Formen, Datentypen, Einheiten, Herkunft und frühen Qualitätskontrollen.

**Zugehörige Vorlesungen**

- **Datenstrukturen**
- **Quellen und Qualität**

## Lernziele

Nach der Bearbeitung können Sie:

- Beobachtungseinheit, Merkmale und Zielwert in tabellarischen Daten eindeutig benennen.
- Eingabematrix X und Zielvektor y mit korrekten Formen erzeugen und dokumentieren.
- CSV-, JSON- und In-Memory-Quellen laden sowie Spalten, Typen, Fehlwerte, Duplikate und Wertebereiche validieren.

## Geprüfte Fähigkeiten

- Listen, Dictionaries, DataFrames, CSV über StringIO und JSON
- X/y-Trennung, Datenwörterbuch, Formen, Einheiten und Wertebereiche
- reproduzierbarer Quellen- und Validierungsbericht

## Hinweise zur Bearbeitung

Dieses Lösungsnotebook enthält dieselben Aufgaben wie das Übungsnotebook sowie vollständige, ausführlich kommentierte Musterlösungen. Bearbeiten Sie nach Möglichkeit zuerst das Übungsnotebook und nutzen Sie dieses Dokument anschließend zur Kontrolle und Vertiefung.

- **Erwarteter Schwierigkeitsgrad:** leicht bis mittel
- Verwenden Sie sprechende Variablennamen und prüfen Sie wichtige Zwischenformen und Wertebereiche.
- Verändern Sie die vorgegebenen Zufalls-Startwerte nur, wenn eine Aufgabe dies ausdrücklich verlangt.
- Interpretieren Sie Ergebnisse fachlich. Eine einzelne Kennzahl ist selten eine vollständige Begründung.
- Alle Aufgaben sind für die kostenlose Google-Colab-Umgebung ausgelegt. Die Datensätze und Modelle sind bewusst klein gehalten. Eine GPU ist nicht erforderlich, kann aber bei einzelnen Deep-Learning-Aufgaben die Laufzeit verkürzen.

## Einrichtung und gemeinsame Datenbasis

Alle Quellen werden als Strings oder Python-Objekte bereitgestellt. So können Sie typische Lade- und Qualitätsprobleme untersuchen, ohne Dateien hochzuladen.

In [ ]:
import json
from io import StringIO

import numpy as np
import pandas as pd

RANDOM_SEED = 42

python_datensaetze = [
    {"anlage_id": "A-101", "temperatur_c": 72.5, "druck_bar": 4.2, "status": "ok", "ausfall": 0},
    {"anlage_id": "A-102", "temperatur_c": 88.0, "druck_bar": 4.9, "status": "warnung", "ausfall": 1},
    {"anlage_id": "A-103", "temperatur_c": 69.0, "druck_bar": 4.0, "status": "OK", "ausfall": 0},
]

csv_text = """anlage_id,temperatur_c,druck_bar,status,ausfall
A-104,75.0,4.4,ok,0
A-105,,4.7,Warnung,1
A-106,140.0,4.5,ok,0
A-106,140.0,4.5,ok,0
"""

json_text = """
[
  {"anlage_id": "A-107", "temperatur_c": 77.0, "druck_bar": 4.3, "status": "ok", "ausfall": 0},
  {"anlage_id": "A-108", "temperatur_c": 81.5, "druck_bar": null, "status": "wartung", "ausfall": 1}
]
"""

soll_spalten = ["anlage_id", "temperatur_c", "druck_bar", "status", "ausfall"]
wertebereiche = {
    "temperatur_c": (0.0, 120.0),
    "druck_bar": (0.0, 10.0),
    "ausfall": (0, 1),
}

print("Einrichtung abgeschlossen.")

### Aufgabe 1: Beobachtungen, Merkmale und Zielwert bestimmen

1. Erzeugen Sie aus `python_datensaetze` einen DataFrame `anlagen_python`.
2. Benennen Sie die Beobachtungseinheit.
3. Legen Sie eine Liste der erklärenden Merkmale und den Zielwert fest.
4. Prüfen Sie, ob jede `anlage_id` eindeutig ist.
5. Geben Sie die ersten Zeilen und Datentypen aus.

In [ ]:
# Die Python-Liste python_datensaetze wurde in der Einrichtungszelle definiert.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Jeder Dictionary-Eintrag wird zu einer Zeile des DataFrames.
anlagen_python = pd.DataFrame(python_datensaetze)

# Eine Beobachtung beschreibt hier eine einzelne Anlage zum dokumentierten Messzeitpunkt.
beobachtungseinheit = "eine Anlage mit einem zusammengehörigen Messsatz"
merkmal_spalten = ["temperatur_c", "druck_bar", "status"]
ziel_spalte = "ausfall"

# is_unique prüft, ob jeder Identifikator nur einmal vorkommt.
ids_eindeutig = anlagen_python["anlage_id"].is_unique

print("Beobachtungseinheit:", beobachtungseinheit)
print("Merkmale:", merkmal_spalten)
print("Zielwert:", ziel_spalte)
print("IDs eindeutig:", ids_eindeutig)
display(anlagen_python)
print(anlagen_python.dtypes)

> **Musterantwort und Interpretation**
>
> Die ID dient primär zur Identifikation und besitzt normalerweise keine fachlich sinnvolle numerische Ordnung. Ein Modell könnte zufällige oder quellenabhängige ID-Muster auswendig lernen, statt physikalische Zusammenhänge zu nutzen.

### Aufgabe 2: Eingabematrix X und Zielvektor y erstellen

Verwenden Sie zunächst nur die beiden numerischen Merkmale `temperatur_c` und `druck_bar`:

1. Erstellen Sie `X` als NumPy-Matrix und `y` als NumPy-Vektor.
2. Geben Sie Form und Datentyp beider Objekte aus.
3. Prüfen Sie, ob die Anzahl der Beobachtungen übereinstimmt.
4. Erstellen Sie zusätzlich einen einzelnen Batch mit den ersten zwei Beobachtungen.

In [ ]:
# Verwenden Sie den DataFrame anlagen_python aus Aufgabe 1.
numerische_merkmale = ["temperatur_c", "druck_bar"]

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# to_numpy() entfernt die pandas-Beschriftung und liefert die numerische Matrix.
X = anlagen_python[numerische_merkmale].to_numpy(dtype=float)
y = anlagen_python["ausfall"].to_numpy(dtype=int)

print("X-Form:", X.shape)
print("X-Datentyp:", X.dtype)
print("y-Form:", y.shape)
print("y-Datentyp:", y.dtype)

# Die Zeilenzahl von X muss genau der Länge von y entsprechen.
assert X.shape[0] == y.shape[0]
assert X.shape[1] == len(numerische_merkmale)

# Ein Batch ist eine Teilmenge von Beobachtungen mit derselben Merkmalsstruktur.
X_batch = X[:2]
y_batch = y[:2]
print("Batch X:", X_batch.shape)
print("Batch y:", y_batch.shape)
print(X_batch)

### Aufgabe 3: Ein Datenwörterbuch dokumentieren

Erstellen Sie einen DataFrame `datenwoerterbuch` mit einer Zeile je Spalte und mindestens diesen Angaben:

- Spaltenname,
- Rolle (`ID`, `Merkmal` oder `Zielwert`),
- fachlicher Datentyp,
- Einheit oder Kategorien,
- erwarteter Wertebereich,
- kurze Bedeutung.

Dokumentieren Sie alle fünf Soll-Spalten.

In [ ]:
datenwoerterbuch = pd.DataFrame()

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Das Wörterbuch dokumentiert fachliche Bedeutung, nicht nur den aktuell erkannten pandas-Datentyp.
datenwoerterbuch = pd.DataFrame(
    [
        {
            "Spalte": "anlage_id",
            "Rolle": "ID",
            "Fachtyp": "kategorialer Identifikator",
            "Einheit_Kategorien": "Format A-###",
            "Erwarteter_Bereich": "eindeutig, nicht leer",
            "Bedeutung": "Identifiziert eine Anlage",
        },
        {
            "Spalte": "temperatur_c",
            "Rolle": "Merkmal",
            "Fachtyp": "numerisch kontinuierlich",
            "Einheit_Kategorien": "°C",
            "Erwarteter_Bereich": "0 bis 120",
            "Bedeutung": "gemessene Betriebstemperatur",
        },
        {
            "Spalte": "druck_bar",
            "Rolle": "Merkmal",
            "Fachtyp": "numerisch kontinuierlich",
            "Einheit_Kategorien": "bar",
            "Erwarteter_Bereich": "0 bis 10",
            "Bedeutung": "gemessener Betriebsdruck",
        },
        {
            "Spalte": "status",
            "Rolle": "Merkmal",
            "Fachtyp": "kategorial nominal",
            "Einheit_Kategorien": "ok, warnung, wartung",
            "Erwarteter_Bereich": "definierte Kategorien",
            "Bedeutung": "gemeldeter Betriebsstatus",
        },
        {
            "Spalte": "ausfall",
            "Rolle": "Zielwert",
            "Fachtyp": "binär kategorial",
            "Einheit_Kategorien": "0 = kein Ausfall, 1 = Ausfall",
            "Erwarteter_Bereich": "0 oder 1",
            "Bedeutung": "zu prognostizierendes Ereignis",
        },
    ]
)

display(datenwoerterbuch)

> **Musterantwort und Interpretation**
>
> Mindestens Messzeitpunkt, Messgerät beziehungsweise Quelle, Erhebungsmethode, Verantwortliche, Datenversion und Lizenz oder Nutzungsrecht sollten dokumentiert werden. Bei wiederholten Messungen ist außerdem entscheidend, ob mehrere Zeilen zu derselben Anlage gehören.

### Aufgabe 4: CSV und JSON laden sowie Quellen kennzeichnen

1. Laden Sie `csv_text` mit `StringIO` in `anlagen_csv`.
2. Laden Sie `json_text` zunächst mit `json.loads` und danach in `anlagen_json`.
3. Ergänzen Sie in allen drei Tabellen eine Spalte `quelle` mit den Werten `python`, `csv` oder `json`.
4. Verbinden Sie die Tabellen zeilenweise zu `anlagen_gesamt` und setzen Sie den Index neu.
5. Prüfen Sie, ob die Spaltenstruktur übereinstimmt.

In [ ]:
# anlagen_python wurde in Aufgabe 1 erstellt.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# StringIO stellt den Text wie eine geöffnete Datei bereit.
anlagen_csv = pd.read_csv(StringIO(csv_text))

# json.loads() wandelt den JSON-Text zunächst in Python-Objekte um.
json_objekte = json.loads(json_text)
anlagen_json = pd.DataFrame(json_objekte)

# Auf Kopien wird die Herkunft jeder Zeile nachvollziehbar dokumentiert.
anlagen_python_quelle = anlagen_python.copy()
anlagen_python_quelle["quelle"] = "python"
anlagen_csv["quelle"] = "csv"
anlagen_json["quelle"] = "json"

# concat() verbindet die Zeilen. ignore_index=True erzeugt einen fortlaufenden Index.
anlagen_gesamt = pd.concat(
    [anlagen_python_quelle, anlagen_csv, anlagen_json],
    ignore_index=True,
)

# Ein Set-Vergleich verhindert, dass eine Quelle unbemerkt andere Spalten liefert.
erwartet_mit_quelle = set(soll_spalten + ["quelle"])
assert set(anlagen_gesamt.columns) == erwartet_mit_quelle

print("Form der Gesamttabelle:", anlagen_gesamt.shape)
display(anlagen_gesamt)

### Aufgabe 5: Qualitätsprobleme systematisch finden

Untersuchen Sie `anlagen_gesamt` und erstellen Sie eine kompakte Ausgabe für:

1. fehlende Soll-Spalten,
2. fehlende Werte je Spalte,
3. vollständig doppelte Zeilen,
4. doppelte `anlage_id`,
5. erkannte pandas-Datentypen,
6. ungültige Werte außerhalb der vorgegebenen Bereiche,
7. unterschiedliche Schreibweisen in `status`.

Verändern Sie die Daten in dieser Aufgabe noch nicht.

In [ ]:
# Verwenden Sie anlagen_gesamt aus Aufgabe 4 und die Vorgaben aus der Setup-Zelle.

# ============================================================
# MUSTERLÖSUNG
# ============================================================

# Spaltenprüfung: Die Differenz zeigt erwartete, aber nicht vorhandene Namen.
fehlende_spalten = sorted(set(soll_spalten) - set(anlagen_gesamt.columns))

# Fehlwerte werden je Spalte gezählt.
fehlwerte = anlagen_gesamt[soll_spalten].isna().sum()

# duplicated() kann vollständige Zeilen oder nur ausgewählte Schlüssel prüfen.
anzahl_zeilenduplikate = int(anlagen_gesamt.duplicated(subset=soll_spalten).sum())
doppelte_ids = anlagen_gesamt[
    anlagen_gesamt.duplicated(subset=["anlage_id"], keep=False)
].sort_values("anlage_id")

# Die Wertebereichsprüfung ignoriert fehlende Werte, markiert aber echte Grenzverletzungen.
bereichsbericht = {}
for spalte, (minimum, maximum) in wertebereiche.items():
    serie = pd.to_numeric(anlagen_gesamt[spalte], errors="coerce")
    ungueltig = serie.notna() & ~serie.between(minimum, maximum)
    bereichsbericht[spalte] = int(ungueltig.sum())

status_varianten = sorted(anlagen_gesamt["status"].dropna().astype(str).unique())

print("Fehlende Soll-Spalten:", fehlende_spalten)
print("\nFehlwerte:")
print(fehlwerte)
print("\nVollständige Duplikate:", anzahl_zeilenduplikate)
print("\nDoppelte IDs:")
display(doppelte_ids)
print("Erkannte Datentypen:")
print(anlagen_gesamt.dtypes)
print("\nGrenzverletzungen:", bereichsbericht)
print("Statusvarianten:", status_varianten)

> **Musterantwort und Interpretation**
>
> Offensichtliche Schreibvarianten können nach einer bestätigten Zuordnung vereinheitlicht und vollständig identische Duplikate können nach Prüfung entfernt werden. Ob 140 °C ein Messfehler, ein gefährlicher realer Zustand oder ein anderer Messkontext ist, erfordert Fachwissen. Auch die Behandlung fehlender Messwerte und mehrfacher IDs hängt von der Beobachtungseinheit ab.

### Aufgabe 6: Integrationsaufgabe: wiederverwendbarer Validierungsbericht

Schreiben Sie eine Funktion `validiere_tabelle(daten, soll_spalten, wertebereiche)`, die einen DataFrame mit mindestens folgenden Prüfungen zurückgibt:

- Spalte vorhanden,
- erkannter Datentyp,
- Anzahl fehlender Werte,
- Anzahl eindeutiger Werte,
- Anzahl Werte außerhalb eines vorgegebenen Bereichs, sofern ein Bereich existiert.

Ergänzen Sie außerdem globale Zeilen für Anzahl vollständiger Duplikate und doppelte IDs. Rufen Sie die Funktion für `anlagen_gesamt` auf.

In [ ]:
def validiere_tabelle(daten, soll_spalten, wertebereiche):
    """Erstellt einen kompakten, tabellarischen Qualitätsbericht."""
    pass

# ============================================================
# MUSTERLÖSUNG
# ============================================================

def validiere_tabelle(daten, soll_spalten, wertebereiche):
    """Erstellt einen kompakten, tabellarischen Qualitätsbericht."""
    berichtszeilen = []

    # Jede erwartete Spalte wird auch dann dokumentiert, wenn sie vollständig fehlt.
    for spalte in soll_spalten:
        vorhanden = spalte in daten.columns
        if not vorhanden:
            berichtszeilen.append(
                {
                    "Prüfobjekt": spalte,
                    "Vorhanden": False,
                    "Datentyp": "nicht vorhanden",
                    "Fehlwerte": np.nan,
                    "Eindeutige_Werte": np.nan,
                    "Außerhalb_Bereich": np.nan,
                }
            )
            continue

        serie = daten[spalte]
        ausserhalb = np.nan
        if spalte in wertebereiche:
            minimum, maximum = wertebereiche[spalte]
            numerisch = pd.to_numeric(serie, errors="coerce")
            # Fehlwerte werden separat gezählt und nicht zusätzlich als Grenzverletzung gewertet.
            ausserhalb = int((numerisch.notna() & ~numerisch.between(minimum, maximum)).sum())

        berichtszeilen.append(
            {
                "Prüfobjekt": spalte,
                "Vorhanden": True,
                "Datentyp": str(serie.dtype),
                "Fehlwerte": int(serie.isna().sum()),
                "Eindeutige_Werte": int(serie.nunique(dropna=True)),
                "Außerhalb_Bereich": ausserhalb,
            }
        )

    # Globale Prüfungen passen nicht zu einer einzelnen Datenspalte, werden aber im selben Bericht ergänzt.
    berichtszeilen.append(
        {
            "Prüfobjekt": "Vollständige Duplikate",
            "Vorhanden": True,
            "Datentyp": "Zeilenprüfung",
            "Fehlwerte": np.nan,
            "Eindeutige_Werte": np.nan,
            "Außerhalb_Bereich": int(daten.duplicated(subset=soll_spalten).sum()),
        }
    )
    berichtszeilen.append(
        {
            "Prüfobjekt": "Doppelte anlage_id",
            "Vorhanden": "anlage_id" in daten.columns,
            "Datentyp": "Schlüsselprüfung",
            "Fehlwerte": np.nan,
            "Eindeutige_Werte": np.nan,
            "Außerhalb_Bereich": int(daten.duplicated(subset=["anlage_id"]).sum())
            if "anlage_id" in daten.columns
            else np.nan,
        }
    )

    return pd.DataFrame(berichtszeilen)


validierungsbericht = validiere_tabelle(anlagen_gesamt, soll_spalten, wertebereiche)
display(validierungsbericht)

> **Musterantwort und Interpretation**
>
> Der Datensatz ist noch nicht modellbereit. Es existieren fehlende Messwerte, ein plausibilitätsbedürftiger Temperaturwert, doppelte Datensätze beziehungsweise IDs und inkonsistente Statusschreibweisen. Vor einer Modellierung müssen Beobachtungseinheit, Duplikatregeln, Grenzverletzungen und Ersatzstrategien fachlich geklärt und anschließend dokumentiert bereinigt werden.

## Abschlusskontrolle

Prüfen Sie vor dem Abschluss:

- Lassen sich alle Zellen in sinnvoller Reihenfolge ausführen?
- Sind Formen, Datentypen, Wertebereiche und Zufalls-Startwerte dokumentiert?
- Wurden Trainings-, Validierungs- und Testinformationen sauber getrennt?
- Sind Diagramme und Kennzahlen beschriftet und fachlich interpretiert?
- Können Sie erklären, warum die gewählten Methoden zur Aufgabenstellung passen?